# SentinelAI

### Notebook académico y técnico del proyecto

**Documento de identificación, explicación y evidencia técnica**

| Campo | Valor |
|---|---|
| Proyecto | SentinelAI |
| Tipo | Plataforma local-first de monitoreo y análisis de ciberseguridad para pequeñas y medianas empresas |
| Autor(es) | *Por completar* |
| Universidad | *Por completar* |
| Profesor(es) | *Por completar* |
| Fecha de entrega | *Por completar* |
| Versión del documento | 1.0 |

> **Nota metodológica.** Este notebook describe el estado **real** del código, los datos y los artefactos existentes en el repositorio. No presenta datos ficticios como reales. Las métricas de inteligencia artificial se identifican explícitamente como experimentos con **datos sintéticos**, por lo que **no demuestran eficacia frente a amenazas reales**. Las funcionalidades incompletas se marcan como *parciales* o *pendientes*.

---

## Tabla de contenido

1. [Identificación del proyecto](#1-identificación-del-proyecto)
2. [Descripción funcional](#2-descripción-funcional)
3. [Arquitectura tecnológica](#3-arquitectura-tecnológica)
4. [Descubrimiento e inventario de red](#4-descubrimiento-e-inventario-de-red)
5. [Inteligencia artificial](#5-inteligencia-artificial)
6. [Datasets y experimentación](#6-datasets-y-experimentación)
7. [Resultados y evidencias](#7-resultados-y-evidencias)
8. [Seguridad y persistencia](#8-seguridad-y-persistencia)
9. [Instalación y ejecución](#9-instalación-y-ejecución)
10. [Conclusiones y trabajo futuro](#10-conclusiones-y-trabajo-futuro)

## 1. Identificación del proyecto

### 1.1 Nombre y tipo

**SentinelAI** es una plataforma **local-first** de monitoreo y evaluación de ciberseguridad empresarial. El propio `README.md` la define como *“Plataforma local de monitoreo y evaluación de ciberseguridad empresarial”*, y la propuesta OpenSpec (`openspec/changes/archive/2026-11-25-sentinelai-platform/proposal.md`) la dirige a pequeñas y medianas empresas que necesitan una herramienta **local, asequible y autocontenida**.

### 1.2 Descripción general

La plataforma integra en una sola aplicación:

- un **agente Windows** que descubre equipos en la subred autorizada y envía resultados al backend;
- un **backend FastAPI** que persiste dispositivos, interfaces de red, software, alertas, vulnerabilidades, eventos y ajustes en **SQLite**;
- dos **modelos de PyTorch entrenados dentro del proyecto** (clasificador de riesgo y transformer de anomalías);
- un **frontend React/TypeScript** con nueve módulos en español;
- despliegue mediante **Docker Compose** con volúmenes persistentes.

El descubrimiento se ejecuta en el **host Windows**, no dentro del contenedor Docker, y no realiza explotación de vulnerabilidades.

### 1.3 Problema que resuelve

Según `openspec/changes/archive/2026-11-25-sentinelai-platform/proposal.md`:

> *“Las pequeñas y medianas empresas necesitan una herramienta local, asequible y autocontenida para descubrir activos de red, evaluar su postura de seguridad y detectar anomalías mediante modelos de IA propios. Las soluciones existentes suelen depender de nube, agentes por host o infraestructura pesada.”*

SentinelAI mantiene los datos en la red local y no requiere servicios externos obligatorios.

### 1.4 Justificación

- Visibilidad de los activos de una red pequeña sin depender de una solucióncloud.
- Persistencia local y auditoría de las operaciones sensibles (borrado/restauración).
- Modelos propios, sin modelos preentrenados sustitutos, con métricas y limitaciones documentadas.
- Interfaz en español y comandos compatibles con Windows PowerShell (`AGENTS.md`).

### 1.5 Objetivo general

Proporcionar una plataforma local que permita **descubrir, inventariar, evaluar y reportar** el estado de seguridad de los equipos de una red autorizada, utilizando datos reales del agente y modelos de IA entrenados en el proyecto.

### 1.6 Objetivos específicos

1. Descubrir dispositivos IPv4 dentro de una subred autorizada mediante ARP/ICMP desde el agente Windows.
2. Persistir dispositivos, interfaces de red, software y su historial sin duplicados.
3. Mostrar el inventario y la información IP en el frontend.
4. Evaluar el riesgo de seguridad de los dispositivos mediante un clasificador propio.
5. Explorar la detección de anomalías de red con un transformer propio.
6. Registrar alertas, vulnerabilidades, eventos y reportes PDF.
7. Desplegar el sistema con Docker Compose preservando los volúmenes de datos.

### 1.7 Alcance

- **Incluido:** subred única autorizada, host Windows con agente, backend/frontend locales, SQLite, dos modelos propios, reportes PDF.
- **No incluido:** explotación activa, descubrimiento fuera de la subred autorizada, detección real de amenazas validada en producción, compatibilidad universal sin pruebas (`README.md`).

### 1.8 Público objetivo

Pequeñas y medianas empresas, docentes y estudiantes que necesiten un entorno reproducible para estudiar arquitectura de seguridad, descubrimiento de red, persistencia y aplicación de modelos de aprendizaje automático.

## 2. Descripción funcional

La aplicación expone nueve módulos en `frontend/src/App.tsx`, todos conectados al backend mediante `frontend/src/services/api.ts` (cliente Axios con `baseURL: '/api'`).

| Módulo | Ruta UI | Funcionalidad real | Estado |
|---|---|---|---|
| **Resumen** (Dashboard) | `/` | Nueve tarjetas con agregados de `GET /api/devices/summary` y `GET /api/alerts/`: confirmados, IPs, interfaces, activos, inactivos, último descubrimiento, alertas, software y riesgo alto. | Implementado; sin gráficos. |
| **Inventario** | `/inventory` | Tabla con IP, MAC, hostname, subred, gateway, estado y última detección; búsqueda por IP/MAC/hostname; filtro de estado; archivado/restauración; detalle con interfaces y software; sondeo de descubrimiento. | Implementado; sin ordenación ni paginación. |
| **Descubrimiento** | integrado en Inventario | El usuario crea una solicitud; el agente Windows la reclama, escanea y envía resultados; la UI sondea el estado cada 2 s. | Implementado; el sondeo no tiene timeout propio. |
| **Topología** | `/topology` | Grafo `@xyflow/react` construido desde dispositivos confirmados, con IP real en cada nodo, filtros y panel de detalle. | Implementado; aristas rotuladas como *“relación lógica estimada”* (no enlaces físicos descubiertos). |
| **Inteligencia Artificial** | `/ai` | Resumen de análisis, selector de dispositivo, botón “Analizar”, evaluación manual, historial y estado técnico de modelos. | Parcial: el clasificador está conectado; el Network Transformer figura como *pendiente de integración de telemetría*. |
| **Vulnerabilidades** | `/vulns` | Listado de CVE/software/severidad, eliminación individual y masiva. | Parcial: backend expone escaneo, pero la UI no tiene botón para invocarlo ni muestra el detalle completo. |
| **Alertas** | `/alerts` | Listado y eliminación individual/masiva. | Parcial: el endpoint `resolve` del backend no tiene cliente ni botón en la UI. |
| **Historial** | `/history` | Eventos con filtro por tipo, búsqueda y eliminación. | Implementado; sin paginación ni filtro por dispositivo en UI. |
| **Reportes** | `/reports` | Generación y descarga de PDF. | Parcial: sin estados de carga/vacío ni listado de reportes previos. |
| **Configuración** | `/settings` | Subred autorizada, intervalo de descubrimiento y agente habilitado, con guardado persistente. | Parcial: validación solo en backend; `getSettings()` no maneja error inicial. |

### 2.1 Relación entre módulos

```text
Agente Windows → POST /api/discovery/agent → dispositivos + interfaces + software (SQLite)
                         ↓
              solicitudes de descubrimiento (claim/complete)
                         ↓
   GET /api/devices/ y /summary → Inventario, Topología y Dashboard
                         ↓
    POST /api/ai/analyze-device/{id} → análisis de riesgo → alertas / historial
                         ↓
       evaluación de vulnerabilidades → caché → listado de vulnerabilidades
                         ↓
                 generación de reporte PDF → descarga
```

### 2.2 Implementado vs. pendiente

- **Implementado y verificado en código:** los nueve módulos, el cliente API con 30 funciones, el detalle de red, el archivado auditado, la generación de PDF y la persistencia SQLite.
- **Parcial/pendiente declarado:** resolución de alertas, botón de escaneo de vulnerabilidades, comprobación manual de disponibilidad, integración visible del Network Transformer, ordenación/paginación del inventario, WebSockets (nginx los prepara, pero no hay código WS en frontend/backend) y las tareas OpenSpec 7.1 y 8.7 de pruebas Docker/persistencia.
- **No utilizado:** `recharts` está en dependencias, pero las visualizaciones reales usan tarjetas, tablas y `@xyflow/react`.

## 3. Arquitectura tecnológica

| Capa | Tecnología real | Evidencia |
|---|---|---|
| Frontend | React 18.3.1, TypeScript 5.9.3, Vite 5.4.21, Tailwind CSS 3.4.19, Axios 1.21.1, React Router 6.30.6, Lucide 0.441.0, `@xyflow/react` 13.0.3 | `frontend/package.json`, `frontend/package-lock.json`, `frontend/src/**` |
| Backend | Python, FastAPI, Pydantic, SQLAlchemy, SQLite, uvicorn | `backend/app/main.py`, `backend/requirements.txt` |
| Persistencia | SQLite (`sqlite:///./data/sentinelai.db` en Docker) + volumen `sqlite_data`; migraciones idempotentes | `backend/app/database.py`, `docker-compose.yml` |
| Agente | Python para Windows, empaquetado con PyInstaller (`sentinel_agent.exe`) | `agent/sentinel_agent.py`, `sentinel_agent.spec` |
| Contenedores | Docker Compose: `backend` (8000) y `frontend`/nginx (80); volúmenes `sqlite_data`, `dataset_data`, `report_data`, `vuln_cache`; bind `./models` | `docker-compose.yml`, `backend/Dockerfile`, `frontend/Dockerfile` |
| IA | PyTorch (modelos definidos en `ai/` y duplicados en `backend/app/ai_models.py`), pesos `.pth` y métricas JSON en `models/` | `ai/**`, `models/**` |

### 3.1 Diagrama de arquitectura (Mermaid)

```mermaid
flowchart LR
    A[Agente Windows<br/>sentinel_agent.exe] -->|ARP / ICMP / PowerShell| B[Backend FastAPI<br/>POST /api/discovery/agent]
    B --> C[(SQLite<br/>SQLAlchemy)]
    C --> D[API REST\n/api/devices, /api/ai, ...]
    D --> E[Frontend React<br/>TypeScript + Vite]
    F[Modelos PyTorch<br/>models/*.pth] -->|torch.load en inference.py| B
    G[Datasets sintéticos<br/>data/*.csv] -->|scripts/train_models.py| F
    H[Docker Compose<br/>volúmenes persistentes] -.-> B
    H -.-> E
```

In [ ]:
# Verificación reproducible de la estructura del repositorio (sin servicios externos).
from pathlib import Path

def encontrar_raiz():
    for candidata in [Path.cwd(), *Path.cwd().parents]:
        if (candidata / "README.md").exists() and (candidata / "backend" / "app" / "main.py").exists():
            return candidata
    return None

RAIZ = encontrar_raiz()
print("Raíz del repositorio:", RAIZ if RAIZ else "No encontrada (ejecuta el notebook desde el repositorio)")

if RAIZ:
    rutas = [
        "README.md", "AGENTS.md", "docker-compose.yml",
        "backend/app/main.py", "backend/app/database.py",
        "frontend/src/App.tsx", "frontend/src/services/api.ts",
        "agent/sentinel_agent.py", "scripts/train_models.py",
        "ai/security_classifier/model.py", "ai/network_transformer/model.py",
        "models/security_classifier.pth", "models/network_transformer.pth",
        "models/security_classifier_metrics.json",
        "models/security_classifier_v3_1_baseline_metrics.json",
        "models/network_transformer_metrics.json",
        "data/security_features_v3_1_corrected.csv",
        "data/network_behavior_v3_1.csv",
    ]
    for r in rutas:
        print(f"{'OK ' if (RAIZ / r).exists() else 'NO '} {r}")

## 4. Descubrimiento e inventario de red

### 4.1 Dónde se ejecuta

El descubrimiento se ejecuta **desde el agente Windows** (`agent/sentinel_agent.py`), dentro de la subred configurada en el backend (`settings.authorized_subnet`, modificable en Configuración). **No se escanea desde el contenedor Docker**. El endpoint `POST /api/discovery/requests` crea una solicitud; el agente la reclama con `POST /api/discovery/requests/claim`, realiza el escaneo, envía el resultado a `POST /api/discovery/agent` y cierra con `POST /api/discovery/requests/{id}/complete`. El agente además reporta `POST /api/agent/heartbeat`. El frontend define `getActiveDiscoveryRequest()` (`GET /api/discovery/requests/active`), pero esa función no es invocada por las pantallas actuales.

### 4.2 Recolección real

- `discover(subnet)`: ARP/ICMP y resolución inversa para obtener IPv4, MAC, fabricante OUI y hostname cuando está disponible.
- `collect_network()`: `Get-NetIPConfiguration` y `Get-CimInstance Win32_OperatingSystem` para obtener todas las interfaces IPv4 del equipo del agente: IP, MAC (normalizada `AA:BB:CC`), prefijo, gateway, DNS, nombre de interfaz, descripción del adaptador y estado (`Up`/`Disconnected`).
- `collect_software()`: inventario de aplicaciones del equipo local, enviado con `software_host_ip` para asociarlo al dispositivo correcto.

### 4.3 Validación y reglas de persistencia

`backend/app/routers/discovery.py` valida cada IP con `is_authorized(ip, subnet)`. Los contadores documentados son:

| Clave devuelta por `POST /api/discovery/agent` | Significado |
|---|---|
| `recibidos` | Dispositivos válidos recibidos (incluye archivados redescubiertos). |
| `created_count` | Altas nuevas. |
| `confirmed_count` | Transiciones a `confirmed=True` (nuevos y existentes sin confirmar). |
| `updated_count` | Existentes no archivados cuyos datos se refrescaban. |
| `archived_count` | Archivados redescubiertos que **no se reactivan**. |
| `rechazados` | Fuera de la subred autorizada. |
| `motivos` | Razones de rechazo por dirección IP. |

Reglas: nuevo válido → confirmado; existente sin confirmar → confirmado; existente confirmado → actualizado; archivado → intacto; datos inválidos/vacíos → no degradan valores previamente válidos. Los dispositivos no reportados se marcan `inactive`, nunca se eliminan físicamente.

### 4.4 Interfaces de red y software

`backend/app/services/network_service.py` sincroniza una tabla normalizada `network_interfaces`: upsert por `(device_id, ip)`, derivación de subred y máscara desde el prefijo con `ipaddress`, dedup, conservación de interfaces que desaparecen (marcadas `inactive`) y selección de la interfaz principal por coincidencia con la IP del dispositivo. `software_service.py` dedup por nombre, adopta filas huérfanas y asocia el software al registro del equipo **incluso si está archivado, sin reactivarlo**.

### 4.5 Diagrama del proceso

```mermaid
flowchart TD
    U[Usuario: Iniciar descubrimiento] --> R[POST /api/discovery/requests]
    R --> A[Agente Windows reclama solicitud]
    A --> S[discover: ARP/ICMP en subred autorizada]
    S --> N[collect_network: interfaces reales de Windows]
    S --> W[collect_software]
    S --> V{IP autorizada?}
    V -- No --> X[rechazados]
    V -- Sí --> C{Existe dispositivo?}
    C -- No --> NEW[crear y confirmar]
    C -- Sí, archivado --> ARC[archivados; no reactivar]
    C -- Sí, activo --> UPD[actualizar; conservar datos válidos]
    NEW --> DB[(SQLite: devices + network_interfaces)]
    UPD --> DB
    ARC --> DB
    W --> DB
    DB --> CMP[POST /api/discovery/requests/{id}/complete]
    CMP --> UI[Inventario / Dashboard / Topología]
```

## 5. Inteligencia artificial

El proyecto entrena sus propios modelos: no se usan modelos preentrenados como sustitutos. `scripts/train_models.py` entrena y persiste los pesos en `models/`; `backend/app/routers/inference.py` carga `state_dict`, aplica el escalador guardado (`scaler_mean`, `scaler_scale`) y devuelve la predicción. No se reentrena automáticamente al iniciar (`README.md`).

### 5.1 Security Classifier

**Archivo:** `ai/security_classifier/model.py` (definición equivalente en `backend/app/ai_models.py`).

- **Arquitectura:** perceptrón multicapa `Linear(9,32) → ReLU → Dropout(0.2) → Linear(32,16) → ReLU → Linear(16,3)`.
- **Entradas (9 características):** `firewall`, `antivirus`, `pending_updates`, `os_age`, `outdated`, `vuln_apps`, `excessive`, `unnecessary`, `unverified` (definidas en `ai/security_classifier/dataset.py` y en el generador V3).
- **Salidas (3 clases):** `bajo`, `medio`, `alto` (etiquetas 0, 1, 2 en `backend/app/routers/inference.py`).
- **Preprocesamiento:** estandarización con media/desviación calculadas **solo en entrenamiento**. El baseline V3.1 agrupa por `family_id` y por huella de características (`_profile`, SHA-256 de los 9 valores) y verifica que no haya solapamiento entre particiones; implementa este agrupamiento directamente en `train_v3_1_baseline.py` (el módulo `ai/common/group_split.py` existe, pero no es importado por los entrenadores).
- **División:** el modelo original usa 70/15/15 estratificado (`ai/security_classifier/train.py`). El baseline V3.1 usa 30 % test y 15 % validación, con un **55 % efectivo de entrenamiento** (`GroupShuffleSplit`, semillas 42 y 43).
- **Entrenamiento:** `ai/security_classifier/train.py` (25 épocas, `Adam(lr=1e-3)`, `CrossEntropyLoss`, batch 64) y `train_v3_1_baseline.py` (hasta 50 épocas, `patience=5`, batch 256, restaura los mejores pesos por `val_acc`). El baseline usa `data/security_features_v3_1_corrected.csv`, semilla 42, 11 épocas y 54,145 s (según `models/security_classifier_v3_1_baseline_metrics.json`).
- **Evaluación:** ver sección 7 con los valores guardados en JSON.
- **Integración:** `POST /api/inference/risk` y, de forma visible en la UI, `POST /api/ai/analyze-device/{id}` (`backend/app/routers/ai.py`, `frontend/src/pages/AI.tsx`). **Dato importante:** esos routers cargan `models/security_classifier.pth` (modelo original de 3.000 filas), **no** el baseline V3.1. `scripts/train_models.py` tampoco ejecuta `train_v3_1_baseline.py`.

**Limitaciones:** los datos son **sintéticos** y las etiquetas se derivan de reglas (`models/security_classifier_metrics.json`: *“Etiquetas derivadas de reglas; no evidencia de riesgo real confirmado”*; el baseline indica *“Datos sintéticos; no evidencia de detección real de amenazas”*). No se afirma detección real de amenazas.

### 5.2 Network Transformer

**Archivo:** `ai/network_transformer/model.py`.

- **Arquitectura:** proyección `Linear(8,32)`, `TransformerEncoder` con `nhead=4` y `num_layers=2` (`batch_first=True`), promedio temporal y clasificador `Linear(32,16) → ReLU → Dropout(0.2) → Linear(16,2)`.
- **Entradas (8 características):** `conn_count`, `conn_freq`, `ports`, `bytes_sent`, `bytes_recv`, `duration`, `activity_var`, `comm_freq` (`ai/network_transformer/dataset.py`).
- **Formato:** acepta `(batch, seq_len, 8)` o `(batch, 8)` (en este último caso se añade una dimensión de secuencia de longitud 1).
- **Inferencia:** `POST /api/inference/anomaly` devuelve `clasificacion` (`normal`/`anómalo`) y `puntuacion_anomalia` mediante softmax.
- **Estado de integración:** el modelo y el endpoint existen, pero la interfaz AI muestra *“Network Transformer: pendiente de integración de telemetría de red”* y no consume `/api/inference/anomaly` ni `/api/telemetry`. El frontend consume sobre todo el clasificador de riesgo.
- **Detalle del checkpoint:** `dim_feedforward=2048` (valor por defecto de `TransformerEncoderLayer`, no explicitado en el código). El archivo `models/network_transformer.pth` contiene las claves del encoder, del clasificador y los vectores `scaler_mean` / `scaler_scale`.
- **Limitaciones:** el dataset es sintético y las métricas guardadas (`accuracy=1.0`) se obtuvieron en ese entorno controlado. La inferencia recibe vectores de 8 valores y, si llegan 2D, la longitud de secuencia es **1** (`unsqueeze(1)`): el promedio temporal no modela serie temporal. El esquema `InferenceIn` no valida la longitud de `features`, y los resultados de `/api/inference/anomaly` no se persisten ni generan alertas. Los datasets V3/V3.1 de ~1 M de filas **nunca** se usaron para entrenar este modelo. **No hay evidencia de detección de anomalías en tráfico real**; la telemetría real del agente no alimenta este modelo.

## 6. Datasets y experimentación

`docs/ai_dataset_v3_audit.md` documenta la corrección V3 frente a V2: se eliminaron duplicados exactos, se equilibraron las clases y se empleó **división por grupos** (`scenario + seed + family_id`), con el escalador ajustado solo en entrenamiento y el test aislado hasta la evaluación final. Todos los datasets listados son **sintéticos generados en el laboratorio**; la auditoría advierte: *“Datos sintéticos no representan tráfico real. Etiquetas sintéticas no son evidencia de amenazas reales.”*

| Archivo (`data/`) | Filas | Columnas | Distribución de `label` | Uso real |
|---|---:|---:|---|---|
| `security_features.csv` | 3.000 | 10 | 0: 985 / 1: 993 / 2: 1.022 | Generador original; reglas de score (`ai/security_classifier/dataset.py`); entrena al modelo usado por el backend. |
| `security_features_expanded.csv` | 1.003.000 | 14 | 0: 339.851 / 1: 362.686 / 2: 300.463 | Versión ampliada V2.1.0; no utilizada por los entrenadores. |
| `security_features_v3_1.csv` | 1.003.000 | 15 | 0: 339.520 / 1: 377.301 / 2: 286.179 | Generación V3.2 con `family_id`; no utilizada por los entrenadores. |
| `security_features_v3_1_corrected.csv` | 1.003.000 | 15 | 0: 258.234 / 1: 178.970 / 2: 565.796 | Dataset del baseline V3.1 (`models/...baseline_metrics.json`). |
| `network_behavior.csv` | 2.000 | 9 | 0: 1.000 / 1: 1.000 | Generador original (`ai/network_transformer/dataset.py`); entrena al modelo usado por el backend. |
| `network_behavior_expanded.csv` | 1.001.791 | 13 | 0: 345.285 / 1: 656.506 | Versión ampliada V2.1.0; no utilizada por los entrenadores. |
| `network_behavior_v3.csv` | No disponible | — | — | No presente en el repositorio actual; la celda de verificación lo marca como ausente. |
| `network_behavior_v3_1.csv` | 1.001.791 | 14 | 0: 498.340 / 1: 503.451 | Generación V3.2 con `family_id`; no utilizada por los entrenadores. |
| `network_behavior_v3_quarantine.csv` | 209 | 9 | 0: 36 / 1: 173 | Filas rechazadas por validación. |
| `network_behavior_quarantine.csv` | 209 | 9 | 0: 36 / 1: 173 | Copia de cuarentena con el mismo SHA-256. |

Los archivos `.hashes.sqlite` asociados sirven para detectar duplicados exactos. **No se modifican los datasets en este notebook**: únicamente se leen.

In [ ]:
# Carga y examen de los datasets disponibles, con rutas relativas y manejo de ausencia.
from pathlib import Path
import pandas as pd

RAIZ = Path.cwd() if (Path.cwd() / "README.md").exists() else next(
    (p for p in Path.cwd().parents if (p / "README.md").exists()), Path.cwd()
)
DATA = RAIZ / "data"

archivos = [
    "security_features.csv", "security_features_expanded.csv",
    "security_features_v3_1.csv", "security_features_v3_1_corrected.csv",
    "network_behavior.csv", "network_behavior_expanded.csv",
    "network_behavior_v3.csv", "network_behavior_v3_1.csv",
    "network_behavior_v3_quarantine.csv", "network_behavior_quarantine.csv",
]

resumen = []
for nombre in archivos:
    ruta = DATA / nombre
    if not ruta.exists():
        resumen.append({"archivo": nombre, "estado": "No disponible"})
        continue
    df = pd.read_csv(ruta)
    fila = {
        "archivo": nombre,
        "registros": len(df),
        "columnas": len(df.columns),
        "target": "label" if "label" in df.columns else "No disponible",
        "clases": str(df["label"].value_counts().sort_index().to_dict()) if "label" in df.columns else "No disponible",
    }
    resumen.append(fila)

tabla = pd.DataFrame(resumen)
with pd.option_context("display.max_colwidth", 80):
    display(tabla.fillna("No disponible"))

## 7. Resultados y evidencias

Los resultados siguientes provienen **exclusivamente** de los archivos JSON guardados en `models/`. Se muestran todos los valores disponibles; cuando un JSON no contiene una métrica, se indica `No disponible`.

### 7.1 Security Classifier — baseline V3.1 (`models/security_classifier_v3_1_baseline_metrics.json`)

| Métrica | Valor real |
|---|---:|
| Accuracy | 0.830195076066038 |
| Precision (clases 0/1/2) | 0.8369087120919824 / 0.5387386357138773 / 0.9167526196113364 |
| Recall (clases 0/1/2) | 0.841845843359466 / 0.5271933751119069 / 0.920490988154852 |
| F1 (clases 0/1/2) | 0.8393700178004798 / 0.5329034812363904 / 0.9186180005226394 |
| Macro F1 | 0.7636304998531699 |
| Weighted F1 | 0.829518593713062 |
| Balanced accuracy | 0.7631767355420749 |
| Mejor accuracy de validación | 0.8307097615499255 |
| Épocas | 11 |
| Tiempo de entrenamiento | 54,145 s |
| Semilla | 42 |

Matriz de confusión (filas = valor real, columnas = predicción):

|  | Pred 0 | Pred 1 | Pred 2 |
|---|---:|---:|---:|
| Real 0 | 65.073 | 11.471 | 754 |
| Real 1 | 11.899 | 28.266 | 13.451 |
| Real 2 | 782 | 12.730 | 156.431 |

### 7.2 Security Classifier — modelo original (`models/security_classifier_metrics.json`)

- Accuracy: **0.9666666666666667**; val_accuracy: **0.9844444444444445**.
- Matriz de confusión 3×3: `[[144,4,0],[2,141,6],[0,3,150]]`.
- Limitación registrada: etiquetas derivadas de reglas, sin evidencia de riesgo real confirmado.

### 7.3 Network Transformer (`models/network_transformer_metrics.json`)

- Accuracy: **1.0**; val_accuracy: **1.0**; matriz `[[150,0],[0,150]]`.
- Limitación registrada: datos sintéticos, sin evidencia de detección real de amenazas.

> Estas cifras son **resultados de laboratorio con datos sintéticos**. No deben interpretarse como rendimiento real de detección de amenazas ni como validación en producción.

In [ ]:
# Lectura de métricas reales guardadas (sin servicios externos).
import json
from pathlib import Path
import pandas as pd

RAIZ = Path.cwd() if (Path.cwd() / "README.md").exists() else next(
    (p for p in Path.cwd().parents if (p / "README.md").exists()), Path.cwd()
)
MODELOS = RAIZ / "models"

nombres = [
    "security_classifier_v3_1_baseline_metrics.json",
    "security_classifier_metrics.json",
    "network_transformer_metrics.json",
]
metricas = {}
for nombre in nombres:
    ruta = MODELOS / nombre
    if ruta.exists():
        with open(ruta, "r", encoding="utf-8") as f:
            metricas[nombre] = json.load(f)
        print(nombre, "-> cargado")
    else:
        print(nombre, "-> No disponible")

if metricas:
    bas = metricas.get("security_classifier_v3_1_baseline_metrics.json", {})
    filas = []
    for clave in ["accuracy", "macro_f1", "weighted_f1", "balanced_accuracy", "best_val_acc", "epochs", "training_time_seconds"]:
        filas.append({"metrica": clave, "valor": bas.get(clave, "No disponible")})
    display(pd.DataFrame(filas))
    print("Dataset declarado:", bas.get("dataset", "No disponible"))
    print("Limitación:", bas.get("limitation", "No disponible"))

In [ ]:
# Matrices de confusión reales extraídas de los JSON (sin datos inventados).
import pandas as pd
from pathlib import Path

RAIZ = Path.cwd() if (Path.cwd() / "README.md").exists() else next(
    (p for p in Path.cwd().parents if (p / "README.md").exists()), Path.cwd()
)
for nombre in ["security_classifier_v3_1_baseline_metrics.json", "security_classifier_metrics.json", "network_transformer_metrics.json"]:
    ruta = RAIZ / "models" / nombre
    if not ruta.exists():
        print(nombre, "-> No disponible")
        continue
    import json
    with open(ruta, "r", encoding="utf-8") as f:
        m = json.load(f)
    matriz = m.get("confusion_matrix")
    if matriz:
        print("\n" + nombre)
        display(pd.DataFrame(matriz, index=[f"Real {i}" for i in range(len(matriz))],
                             columns=[f"Pred {i}" for i in range(len(matriz))]))

In [ ]:
# Distribución de clases del dataset corregido V3.1 (solo lectura).
from pathlib import Path
import pandas as pd

RAIZ = Path.cwd() if (Path.cwd() / "README.md").exists() else next(
    (p for p in Path.cwd().parents if (p / "README.md").exists()), Path.cwd()
)
csv = RAIZ / "data" / "security_features_v3_1_corrected.csv"
if not csv.exists():
    print("Dataset no disponible: data/security_features_v3_1_corrected.csv")
else:
    df = pd.read_csv(csv)
    dist = df["label"].value_counts().sort_index()
    display(pd.DataFrame({"registros": dist, "porcentaje": (dist / len(df) * 100).round(2)}))
    try:
        import matplotlib.pyplot as plt
        dist.plot(kind="bar", color="#3b82f6", title="Distribución de clases (datos sintéticos)")
        plt.xlabel("Clase"); plt.ylabel("Registros"); plt.tight_layout(); plt.show()
    except ImportError:
        print("matplotlib no disponible: se omite la gráfica; la tabla anterior contiene los datos reales.")

In [ ]:
# Artefactos de modelos: presencia y tamaño real de los archivos .pth y JSON.
from pathlib import Path

RAIZ = Path.cwd() if (Path.cwd() / "README.md").exists() else next(
    (p for p in Path.cwd().parents if (p / "README.md").exists()), Path.cwd()
)
for nombre in sorted((RAIZ / "models").glob("*")):
    print(f"{nombre.name:55s} {nombre.stat().st_size:>12,} bytes")

### 7.4 Coherencia entre artefactos: contradicciones detectadas

La lectura del código y de los artefactos permite identificar estas diferencias que deben declararse en una presentación académica:

| Aspecto | Situación real |
|---|---|
| Modelo desplegado | El backend carga `security_classifier.pth` (3.000 filas sintéticas), **no** `security_classifier_v3_1_baseline.pth` (1.003.000 filas). El baseline V3.1 existe como evidencia de experimentación, no como modelo en producción. |
| División de datos | El JSON del modelo original declara 70/15/15; el entrenador V3.1 produce 55/15/30 y su JSON no incluye el campo `split`. |
| Distribución objetivo V3 | `docs/ai_dataset_v3_audit.md` propone 33/33/34 en seguridad; el dataset corregido tiene 25,75/17,84/56,41. En red sí se cumple aproximadamente 50/50. |
| Trazabilidad del dataset corregido | No hay en el repositorio un script que escriba explícitamente `data/security_features_v3_1_corrected.csv`; solo es citado por el entrenador y por las métricas. |
| Métricas perfectas | `accuracy=1.0` del transformer se debe a dos gaussianas sintéticas claramente separables y a una secuencia de longitud 1. |

Estas contradicciones **no se ocultan**: forman parte del estado real del proyecto y se recogen como trabajo futuro en la sección 10.

In [ ]:
# ¿Qué pesos .pth referencia realmente el backend? (solo lectura de código).
from pathlib import Path

RAIZ = Path.cwd() if (Path.cwd() / "README.md").exists() else next(
    (p for p in Path.cwd().parents if (p / "README.md").exists()), Path.cwd()
)
fuentes = [
    "backend/app/routers/ai.py",
    "backend/app/routers/inference.py",
    "backend/app/routers/training.py",
    "scripts/train_models.py",
]
for rel in fuentes:
    ruta = RAIZ / rel
    if not ruta.exists():
        print(rel, "-> No disponible")
        continue
    texto = ruta.read_text(encoding="utf-8", errors="replace")
    referencias = sorted({linea.strip() for linea in texto.splitlines() if ".pth" in linea})
    print("\n" + rel)
    print("\n".join(referencias) if referencias else "Sin referencias directas a .pth")

In [ ]:
# Mapa de calor de la matriz de confusión real del baseline V3.1 (datos del JSON).
import json
from pathlib import Path

RAIZ = Path.cwd() if (Path.cwd() / "README.md").exists() else next(
    (p for p in Path.cwd().parents if (p / "README.md").exists()), Path.cwd()
)
ruta = RAIZ / "models" / "security_classifier_v3_1_baseline_metrics.json"
if not ruta.exists():
    print("Métricas no disponibles: models/security_classifier_v3_1_baseline_metrics.json")
else:
    with open(ruta, "r", encoding="utf-8") as f:
        matriz = json.load(f).get("confusion_matrix")
    if not matriz:
        print("El JSON no contiene matriz de confusión; no se genera gráfica.")
    else:
        try:
            import matplotlib.pyplot as plt
            fig, ax = plt.subplots(figsize=(5, 4))
            imagen = ax.imshow(matriz, cmap="Blues")
            for i, fila in enumerate(matriz):
                for j, valor in enumerate(fila):
                    ax.text(j, i, f"{valor:,}", ha="center", va="center",
                            color="white" if valor > max(map(max, matriz)) / 2 else "#0b1220")
            ax.set_xticks(range(3), ["bajo", "medio", "alto"])
            ax.set_yticks(range(3), ["bajo", "medio", "alto"])
            ax.set_xlabel("Predicción"); ax.set_ylabel("Valor real")
            ax.set_title("Matriz de confusión — baseline V3.1 (datos sintéticos)")
            fig.colorbar(imagen, ax=ax); plt.tight_layout(); plt.show()
        except ImportError:
            print("matplotlib no disponible; la matriz ya se mostró en forma de tabla.")

### 7.5 Evidencia operativa del backend (consulta opcional)

**Requisito:** backend en ejecución (`docker compose up -d`). La celda siguiente consulta tres endpoints reales y muestra **únicamente** lo que devuelve la API. Si el servicio no está disponible, imprime un mensaje informativo y no falla. No se fijan cifras en el texto porque los contadores cambian con cada descubrimiento; la evidencia es la respuesta obtenida al ejecutar la celda.

- `GET /api/devices/summary`: totales de dispositivos confirmados/activos/inactivos/archivados, interfaces, IPs, software y último descubrimiento.
- `GET /api/agent/status`: conexión del agente y última actividad.
- `GET /api/ai/summary`: número de análisis y desglose de resultados.

In [ ]:
# Evidencia operativa real (REQUIERE backend; sin backend muestra mensaje y no falla).
import json
import urllib.request

BASE = "http://localhost:8000"
for endpoint in ["/api/devices/summary", "/api/agent/status", "/api/ai/summary"]:
    try:
        with urllib.request.urlopen(BASE + endpoint, timeout=3) as resp:
            datos = json.loads(resp.read().decode("utf-8"))
        print(f"\n{endpoint}")
        print(json.dumps(datos, ensure_ascii=False, indent=2))
    except Exception:
        print(f"\n{endpoint}: no disponible (requiere 'docker compose up -d').")

## 8. Seguridad y persistencia

- **Subred autorizada:** el backend valida cada IP contra `authorized_subnet` (`backend/app/discovery/discovery.py`, `settings_service.py`); fuera de subred se contabiliza como rechazado. No hay explotación.
- **Confirmación:** `confirmed=True` solo para dispositivos validados por el agente; los registros históricos sin confirmar no se muestran en el inventario por defecto.
- **Control de duplicados:** upsert por IP en `devices` y por `(device_id, ip)` en `network_interfaces`; dedup de software por nombre con adopción de huérfanos.
- **Archivado:** borrado lógico mediante `deleted_at`, con `POST /api/devices/{id}/restore` para restauración explícita. El descubrimiento **nunca reactiva** un dispositivo archivado.
- **Auditoría:** `deletion_audit` registra `soft_delete`, `bulk_soft_delete` y `restore` con fecha y detalles (`backend/app/services/audit_service.py`).
- **Retención:** el scheduler de limpieza (`backend/app/services/cleanup_service.py`) actúa sobre eventos/alertas antiguos; los dispositivos archivados y su histórico se conservan.
- **Persistencia:** SQLite en `./data/sentinelai.db` dentro del contenedor; volumen `sqlite_data` en `docker-compose.yml`. Los archivos `backend/*.db` locales son de pruebas, no la BD de producción.
- **Docker:** volúmenes `sqlite_data`, `dataset_data`, `report_data`, `vuln_cache` y bind de `./models`; healthchecks en backend y frontend; política de **no eliminar volúmenes** (`docker compose down -v` prohibido por `AGENTS.md`).
- **Credenciales:** el agente usa `API_KEY` y `SENTINEL_API_URL` desde `agent/.env.local` (no versionado). Este notebook **no incluye claves, contraseñas, tokens ni direcciones privadas**.

In [ ]:
# Comprobación opcional del backend (REQUIERE SERVICIO externo).
# Si no está ejecutándose, la celda informa y no falla.
import json
import urllib.request

try:
    with urllib.request.urlopen("http://localhost:8000/api/health", timeout=3) as resp:
        datos = json.loads(resp.read().decode("utf-8"))
    print("Backend disponible. estado:", datos.get("status", "No disponible"))
except Exception:
    print("Backend no disponible en localhost:8000 (requiere 'docker compose up -d').")

In [ ]:
# Comprobación opcional de Docker Compose (REQUIERE Docker Desktop en ejecución).
import shutil
import subprocess

if not shutil.which("docker"):
    print("Docker no disponible en el PATH; se omite la comprobación.")
else:
    try:
        salida = subprocess.run(
            ["docker", "compose", "ps", "--format", "{{.Name}} {{.Status}}"],
            capture_output=True, text=True, timeout=15, check=True,
        )
        print(salida.stdout.strip() or "No hay contenedores del proyecto en ejecución.")
    except Exception as exc:
        print("No se pudo consultar Docker Compose:", type(exc).__name__)

## 9. Instalación y ejecución

Comandos reales para **Windows PowerShell**, extraídos de `README.md` y `docker-compose.yml`. No se incluyen comandos destructivos ni eliminación de volúmenes.

### 9.1 Requisitos (`README.md`)

```powershell
# Docker Desktop (Windows 11), Python 3.11 y Node.js 20
```

### 9.2 Despliegue con Docker Compose (recomendado)

```powershell
copy .env.example .env
docker compose build
docker compose up -d
docker compose ps
docker compose logs -f
# Dashboard: http://localhost
# API: http://localhost:8000/docs
docker compose down          # detener SIN borrar volúmenes
```

### 9.3 Desarrollo local — backend

```powershell
cd backend
python -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -r requirements.txt
uvicorn app.main:app --reload
```

### 9.4 Desarrollo local — frontend

```powershell
cd frontend
npm install
npm run dev
```

### 9.5 Agente Windows

```powershell
# Ejecución directa desde el host Windows
python agent/sentinel_agent.py

# Ejecutable empaquetado (README.md)
pyinstaller --onefile --name sentinel_agent --distpath agent\dist --workpath agent\build agent\sentinel_agent.py
.\agent\dist\sentinel_agent.exe --daemon

# Instalación como tarea programada
.\scripts\install_agent_task.ps1
```

La configuración sensible del agente se coloca en `agent\.env.local` (no versionado): `SENTINEL_API_URL`, `AUTHORIZED_SUBNET` y `API_KEY`.

### 9.6 Modelos y pruebas

```powershell
# Entrenamiento explícito (los pesos se guardan en models/)
python scripts/train_models.py

# Pruebas del backend
cd backend
python -m pytest -q
```

> No se reentrena automáticamente al iniciar si existen pesos válidos (`README.md`, `AGENTS.md`).

## 10. Conclusiones y trabajo futuro

### 10.1 Aportes del proyecto

- Integración de descubrimiento, inventario, riesgo, vulnerabilidades, alertas, historial y reportes en una plataforma local con interfaz en español.
- Persistencia real en SQLite con migraciones idempotentes, auditoría de archivado/restauración y respeto del histórico.
- Inventario de red normalizado: múltiples interfaces por dispositivo, sin duplicados y sin perder datos ante recolecciones incompletas.
- Dos modelos propios con pesos, métricas y limitaciones documentadas.

### 10.2 Funcionalidades implementadas

Los nueve módulos de la interfaz están montados y consumen endpoints reales; el agente Windows descubre y envía dispositivos, interfaces y software; el backend persiste y expone los datos; Docker Compose despliega y mantiene volúmenes; el clasificador de riesgo está integrado en el flujo de análisis.

### 10.3 Resultados obtenidos

- Métricas experimentales reales (sección 7), obtenidas con datasets **sintéticos**.
- Evidencia de persistencia y visualización de IP/interfaces en el inventario (sección 4).
- Pruebas automáticas del backend (44 pruebas en la ejecución de referencia; `backend/tests/test_api.py`).

### 10.4 Limitaciones técnicas

- Los resultados de IA **no validan detección de amenazas reales**: los datos son sintéticos y las etiquetas derivan de reglas.
- El backend despliega el clasificador **original** (3.000 filas); el baseline V3.1 (1.003.000 filas, accuracy 0,8302) queda como evidencia experimental no integrada.
- Existen discrepancias documentales reales: el split 70/15/15 del JSON original frente al 55/15/30 del baseline, y la distribución objetivo 33/33/34 frente al 25,75/17,84/56,41 del dataset corregido (sección 7.4).
- El Network Transformer no recibe telemetría real en la aplicación y su integración visible está pendiente; el endpoint de telemetría almacena las 8 variables, pero no invoca al modelo ni persiste el resultado. Además, con entradas 2D la longitud de secuencia es 1 y el esquema de inferencia no valida el número de características.
- Topología muestra relaciones lógicas estimadas, no enlaces físicos descubiertos.
- Inventario sin ordenación/paginación; Reportes sin estados de carga; Configuración sin validación client-side.
- No hay WebSockets activos: se usa sondeo HTTP.
- `recharts` está declarado pero no se utiliza.

### 10.5 Trabajo pendiente y mejoras futuras

1. Resolución/reconocimiento de alertas desde la UI y botón de escaneo de vulnerabilidades.
2. Comprobación de disponibilidad desde la interfaz (el endpoint backend ya existe).
3. Integración de telemetría real con el Network Transformer y validación con datos capturados en laboratorio.
4. Ordenación, paginación y filtros avanzados en el inventario.
5. Completar y marcar las tareas OpenSpec 7.1 y 8.7 de pruebas Docker/persistencia. El cambio `sentinelai-platform` fue archivado el 2026-11-25 en `openspec/changes/archive/2026-11-25-sentinelai-platform/` con esas dos tareas pendientes; sus 14 specs delta se sincronizaron a `openspec/specs/`.
6. Unificar idioma de la documentación OpenSpec al español y ampliar los estados vacíos/carga en todos los módulos.

### 10.6 Conclusión general

SentinelAI es una plataforma funcional y reproducible en su alcance local: **convierte datos reales del agente Windows en inventario persistente y visible**, aplica un clasificador de riesgo propio y genera trazabilidad mediante alertas, eventos y reportes. Su componente de IA es **experimental**: las métricas son rigorosas dentro del laboratorio sintético, pero todavía no constituyen evidencia de eficacia contra amenazas reales. El trabajo futuro debe centrarse en cerrar las integraciones pendientes y validar los modelos con telemetría auténtica.